# LumenY — Feature Engineering (Sub-hourly)

Builds the full feature matrix from processed OHLCV data across all timeframes.

**Architecture:** 5m as base index, features computed from 5m, 15m, 1H, 4H, 1D timeframes and merged into one row per 5m timestamp.

**Labels:** 5min (1 bar), 15min (3 bars), 1H (12 bars) — all computed from 5m close prices.

**Output:** One combined parquet for all pairs ready for v3 model training.

In [ ]:
import pandas as pd
import numpy as np
import pandas_ta as ta
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('../backend/data/processed')
FEATURES_DIR  = Path('../backend/data/features_2')
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ['EURUSD', 'GBPUSD', 'USDJPY', 'USDCHF', 'AUDUSD', 'USDCAD', 'NZDUSD']

print('Ready.')
print(f'Features will be saved to: {FEATURES_DIR.resolve()}')

## 1. Feature Engineering Functions

Same functions as v2 — each takes a OHLCV DataFrame and returns features with a `_{tf}` suffix.

In [ ]:
def compute_features(df: pd.DataFrame, tf: str) -> pd.DataFrame:
    """
    Compute all TA features for a given OHLCV DataFrame.
    Returns a DataFrame with all features, columns suffixed with _{tf}.
    All features use only past data — no lookahead.
    """
    feat = pd.DataFrame(index=df.index)
    o, h, l, c = df['open'], df['high'], df['low'], df['close']
    
    # -- RETURNS & MOMENTUM --
    for n in [1, 3, 6, 12, 24, 48]:
        feat[f'log_ret_{n}'] = np.log(c / c.shift(n))
    
    feat['rsi_14'] = ta.rsi(c, length=14)
    feat['rsi_28'] = ta.rsi(c, length=28)
    feat['rsi_slope'] = feat['rsi_14'] - feat['rsi_14'].shift(3)
    
    price_higher = (c > c.shift(5)).astype(int)
    rsi_higher   = (feat['rsi_14'] > feat['rsi_14'].shift(5)).astype(int)
    feat['rsi_divergence'] = (price_higher != rsi_higher).astype(int)
    
    macd = ta.macd(c, fast=12, slow=26, signal=9)
    if macd is not None:
        feat['macd']        = macd.iloc[:, 0]
        feat['macd_signal'] = macd.iloc[:, 2]
        feat['macd_hist']   = macd.iloc[:, 1]
    
    for n in [20, 50, 200]:
        ma = ta.sma(c, length=n)
        feat[f'dist_ma_{n}'] = (c - ma) / c
    
    ema_fast = ta.ema(c, length=9)
    ema_slow = ta.ema(c, length=21)
    feat['ema_cross'] = (ema_fast > ema_slow).astype(int)
    feat['ema_dist']  = (ema_fast - ema_slow) / c

    # -- VOLATILITY --
    atr_14 = ta.atr(h, l, c, length=14)
    atr_28 = ta.atr(h, l, c, length=28)
    feat['atr_14_norm'] = atr_14 / c
    feat['atr_28_norm'] = atr_28 / c
    feat['atr_ratio'] = atr_14 / atr_28
    
    log_ret = np.log(c / c.shift(1))
    feat['rvol_12']  = log_ret.rolling(12).std()
    feat['rvol_24']  = log_ret.rolling(24).std()
    feat['rvol_48']  = log_ret.rolling(48).std()
    feat['rvol_ratio'] = feat['rvol_12'] / feat['rvol_48']
    
    feat['hl_range'] = (h - l) / c
    
    bb = ta.bbands(c, length=20, std=2)
    if bb is not None:
        bb_upper = bb.iloc[:, 0]
        bb_mid   = bb.iloc[:, 1]
        bb_lower = bb.iloc[:, 2]
        feat['bb_width']    = (bb_upper - bb_lower) / bb_mid
        feat['bb_position'] = (c - bb_lower) / (bb_upper - bb_lower + 1e-10)
    
    # -- MARKET STRUCTURE & KEY LEVELS --
    for n in [20, 50]:
        feat[f'dist_high_{n}'] = (c - h.rolling(n).max()) / c
        feat[f'dist_low_{n}']  = (c - l.rolling(n).min()) / c
    
    feat['breakout_20'] = (c > h.shift(1).rolling(20).max()).astype(int)
    feat['breakdown_20'] = (c < l.shift(1).rolling(20).min()).astype(int)
    
    def rolling_slope(series, n):
        slopes = series.copy() * np.nan
        x = np.arange(n)
        for i in range(n, len(series)):
            y = series.iloc[i-n:i].values
            if not np.any(np.isnan(y)):
                slopes.iloc[i] = np.polyfit(x, y, 1)[0] / series.iloc[i]
        return slopes
    
    feat['trend_slope_20'] = rolling_slope(c, 20)
    
    body  = abs(c - o)
    range_ = h - l + 1e-10
    feat['body_ratio']       = body / range_
    feat['upper_wick_ratio'] = (h - pd.concat([c, o], axis=1).max(axis=1)) / range_
    feat['lower_wick_ratio'] = (pd.concat([c, o], axis=1).min(axis=1) - l) / range_
    
    adx = ta.adx(h, l, c, length=14)
    if adx is not None:
        feat['adx'] = adx.iloc[:, 0]
    
    feat.columns = [f'{col}_{tf}' for col in feat.columns]
    
    return feat


print('Feature function ready.')

In [ ]:
def compute_time_features(df: pd.DataFrame) -> pd.DataFrame:
    feat = pd.DataFrame(index=df.index)
    
    feat['hour'] = df.index.hour
    feat['minute'] = df.index.minute
    
    feat['session_asian']  = ((df.index.hour >= 0)  & (df.index.hour < 8)).astype(int)
    feat['session_london'] = ((df.index.hour >= 8)  & (df.index.hour < 16)).astype(int)
    feat['session_ny']     = ((df.index.hour >= 13) & (df.index.hour < 21)).astype(int)
    feat['session_overlap'] = ((df.index.hour >= 13) & (df.index.hour < 16)).astype(int)
    
    feat['day_of_week'] = df.index.dayofweek
    feat['is_monday']   = (df.index.dayofweek == 0).astype(int)
    feat['is_friday']   = (df.index.dayofweek == 4).astype(int)
    
    feat['month'] = df.index.month
    
    return feat


print('Time feature function ready.')

In [ ]:
def compute_crossTF_features(feat_1h: pd.DataFrame, feat_4h: pd.DataFrame, feat_1d: pd.DataFrame) -> pd.DataFrame:
    feat = pd.DataFrame(index=feat_1h.index)
    
    if 'ema_cross_1H' in feat_1h.columns and 'ema_cross_4H' in feat_4h.columns:
        feat['trend_align_1h_4h'] = (feat_1h['ema_cross_1H'] == feat_4h['ema_cross_4H']).astype(int)
    
    if 'ema_cross_4H' in feat_4h.columns and 'ema_cross_1D' in feat_1d.columns:
        feat['trend_align_4h_1d'] = (feat_4h['ema_cross_4H'] == feat_1d['ema_cross_1D']).astype(int)
    
    if all(c in feat.columns for c in ['trend_align_1h_4h', 'trend_align_4h_1d']):
        feat['full_confluence'] = (feat['trend_align_1h_4h'] & feat['trend_align_4h_1d']).astype(int)
    
    if 'atr_ratio_1H' in feat_1h.columns and 'atr_ratio_1D' in feat_1d.columns:
        feat['vol_expansion'] = (feat_1h['atr_ratio_1H'] > feat_1d['atr_ratio_1D']).astype(int)
    
    if 'rsi_14_1H' in feat_1h.columns and 'rsi_14_4H' in feat_4h.columns:
        feat['rsi_align_1h_4h'] = np.sign(feat_1h['rsi_14_1H'] - 50) == np.sign(feat_4h['rsi_14_4H'] - 50)
        feat['rsi_align_1h_4h'] = feat['rsi_align_1h_4h'].astype(int)
    
    scores = []
    for col, df_ in [('rsi_14_1H', feat_1h), ('rsi_14_4H', feat_4h), ('rsi_14_1D', feat_1d)]:
        if col in df_.columns:
            scores.append((df_[col] > 50).astype(int))
    if scores:
        feat['momentum_confluence'] = sum(scores)
    
    return feat


print('Cross-TF feature function ready.')

## 2. Build Feature Matrix Per Pair (5m base index)

Same features as v2, but base index is now 5m instead of 1H.
Higher timeframe features are forward-filled down to 5m resolution.

In [ ]:
def build_feature_matrix_subhourly(pair: str) -> pd.DataFrame:
    """
    Build the full feature matrix for a given pair.
    Base index = 5m candles.
    Features from 5m, 15m, 1H, 4H, 1D — all aligned to 5m timestamps.
    """
    print(f'  Loading data...')
    
    dfs = {}
    for tf in ['5m', '15m', '1H', '4H', '1D']:
        path = PROCESSED_DIR / f'{pair}_{tf}.parquet'
        if path.exists():
            dfs[tf] = pd.read_parquet(path)
        else:
            print(f'  WARNING: Missing {tf} data for {pair}')
    
    # Base index is 5m
    base = dfs['5m'].copy()
    
    print(f'  Computing features...')
    
    # Compute features for each timeframe
    feat_5m  = compute_features(dfs['5m'],  '5m')
    feat_15m = compute_features(dfs['15m'], '15m') if '15m' in dfs else None
    feat_1h  = compute_features(dfs['1H'],  '1H')  if '1H'  in dfs else None
    feat_4h  = compute_features(dfs['4H'],  '4H')  if '4H'  in dfs else None
    feat_1d  = compute_features(dfs['1D'],  '1D')  if '1D'  in dfs else None
    
    # Time features (based on 5m index)
    feat_time = compute_time_features(base)
    
    print(f'  Aligning timeframes to 5m index...')
    
    # Start with 5m features (same index)
    all_features = feat_5m.copy()
    
    # Merge 15m features — forward fill to 5m (each 15m value persists for 3 bars)
    if feat_15m is not None:
        feat_15m_5m = feat_15m.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_15m_5m, how='left')
    
    # Merge 1H features — forward fill to 5m (each 1H value persists for 12 bars)
    if feat_1h is not None:
        feat_1h_5m = feat_1h.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_1h_5m, how='left')
    
    # Merge 4H features — forward fill to 5m
    if feat_4h is not None:
        feat_4h_5m = feat_4h.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_4h_5m, how='left')
    
    # Merge 1D features — forward fill to 5m
    if feat_1d is not None:
        feat_1d_5m = feat_1d.reindex(all_features.index, method='ffill')
        all_features = all_features.join(feat_1d_5m, how='left')
    
    # Cross-TF features (computed on native TFs, then aligned to 5m)
    if feat_1h is not None and feat_4h is not None and feat_1d is not None:
        feat_1h_aligned = feat_1h.reindex(all_features.index, method='ffill')
        feat_4h_aligned = feat_4h.reindex(all_features.index, method='ffill')
        feat_1d_aligned = feat_1d.reindex(all_features.index, method='ffill')
        feat_cross = compute_crossTF_features(feat_1h_aligned, feat_4h_aligned, feat_1d_aligned)
        all_features = all_features.join(feat_cross, how='left')
    
    # Time features
    all_features = all_features.join(feat_time, how='left')
    
    # Pair identity
    pair_map = {p: i for i, p in enumerate(PAIRS)}
    all_features['pair_id'] = pair_map[pair]
    
    print(f'  Feature matrix shape: {all_features.shape}')
    
    return all_features


print('Feature matrix builder ready (5m base index).')

## 2b. Cross-Pair Correlation Features

Rolling correlation between each pair and all others, computed on 1H closes then forward-filled to 5m.

In [ ]:
# Build features for all pairs first
pair_features = {}

for pair in PAIRS:
    print(f'\nBuilding features for {pair}...')
    try:
        pair_features[pair] = build_feature_matrix_subhourly(pair)
    except Exception as e:
        print(f'  ERROR: {e}')
        import traceback
        traceback.print_exc()

print('\nAll pair features built.')

In [ ]:
# Cross-pair correlations computed on 1H closes, then forward-filled to 5m index
print('Computing cross-pair correlations...')

closes_1h = {}
for pair in PAIRS:
    df_1h = pd.read_parquet(PROCESSED_DIR / f'{pair}_1H.parquet')
    closes_1h[pair] = df_1h['close']

close_df = pd.DataFrame(closes_1h)

for window, w_name in [(24, '24H'), (168, '1W')]:
    print(f'  Computing {w_name} rolling correlations...')
    rolling_corr = close_df.rolling(window).corr()
    
    for pair in PAIRS:
        if pair not in pair_features:
            continue
        feat_5m_index = pair_features[pair].index
        
        for other_pair in PAIRS:
            if other_pair == pair:
                continue
            col_name = f'corr_{other_pair}_{w_name}'
            try:
                corr_series = rolling_corr.xs(pair, level=1)[other_pair]
                # Forward-fill 1H correlations to 5m index
                pair_features[pair][col_name] = corr_series.reindex(feat_5m_index, method='ffill')
            except Exception as e:
                pair_features[pair][col_name] = np.nan
    
    print(f'  {w_name} correlations added.')

# Verify
sample = pair_features['EURUSD']
corr_cols = [c for c in sample.columns if c.startswith('corr_')]
print(f'\nCorrelation features added: {len(corr_cols)}')
print(f'Example columns: {corr_cols[:4]}')
print('Cross-pair correlations complete.')

## 3. Compute Labels

Forward log returns computed from **5m close prices**.
- `label_5min` = 1 bar ahead (5 minutes)
- `label_15min` = 3 bars ahead (15 minutes)
- `label_1H` = 12 bars ahead (1 hour)

Same formula as v2: `log(close[t+N] / close[t])`

In [ ]:
HORIZONS = {
    '5min': 1,    # 1 × 5m bar = 5 minutes
    '15min': 3,   # 3 × 5m bars = 15 minutes
    '1H': 12,     # 12 × 5m bars = 1 hour
}

def compute_labels_subhourly(pair: str, feat_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute forward log returns for each horizon.
    Uses 5m close prices (same base index as features).
    """
    df_5m = pd.read_parquet(PROCESSED_DIR / f'{pair}_5m.parquet')
    close = df_5m['close'].reindex(feat_df.index)
    
    labels = pd.DataFrame(index=feat_df.index)
    
    for horizon_name, n_bars in HORIZONS.items():
        labels[f'label_{horizon_name}'] = np.log(close.shift(-n_bars) / close)
    
    return labels


print('Label function ready.')
print(f'Horizons: {HORIZONS}')

In [ ]:
# Combine all pairs into one dataset with features + labels
all_dfs = []

for pair in PAIRS:
    print(f'Processing labels for {pair}...')
    
    if pair not in pair_features:
        print(f'  Missing features, skipping.')
        continue
    
    feat_df = pair_features[pair]
    label_df = compute_labels_subhourly(pair, feat_df)
    
    combined = feat_df.join(label_df, how='left')
    combined['pair'] = pair
    
    all_dfs.append(combined)
    print(f'  {pair}: {combined.shape}')

# Combine all pairs
df_all = pd.concat(all_dfs, axis=0)
df_all = df_all.sort_index()

print(f'\nCombined dataset shape: {df_all.shape}')
print(f'Date range: {df_all.index[0]} -> {df_all.index[-1]}')
print(f'Pairs: {df_all["pair"].unique()}')

In [ ]:
# Drop rows where any label is NaN (end of dataset — no future data)
label_cols = [f'label_{h}' for h in HORIZONS.keys()]
df_all = df_all.dropna(subset=label_cols)

# Save combined dataset
out_path = FEATURES_DIR / 'all_pairs_features_labels_subhourly.parquet'
df_all.to_parquet(out_path)

print(f'Combined dataset saved: {df_all.shape}')
print(f'Features: {df_all.shape[1] - len(label_cols) - 1} columns')
print(f'Labels: {label_cols}')
print(f'\nLabel statistics:')
df_all[label_cols].describe()

## 4. Validate Features

In [ ]:
# Check feature count matches v2 (should be 208 features)
feat_cols = [c for c in df_all.columns if c not in label_cols + ['pair']]

nan_ratio = df_all[feat_cols].isnull().mean().sort_values(ascending=False)
print(f'Total features: {len(feat_cols)}')
print(f'\nFeatures with >10% NaN:')
print(nan_ratio[nan_ratio > 0.1])
print(f'\nFeatures with 0% NaN: {(nan_ratio == 0).sum()}')

In [ ]:
# Label distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#080c14')

for ax, horizon in zip(axes.flatten(), HORIZONS.keys()):
    col = f'label_{horizon}'
    data = df_all[col].dropna()
    
    p1, p99 = data.quantile(0.01), data.quantile(0.99)
    data_clipped = data.clip(p1, p99)
    
    ax.hist(data_clipped, bins=100, color='#4fc3f7', alpha=0.7, edgecolor='none')
    ax.axvline(0, color='#ff4757', linewidth=1.5, linestyle='--')
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='white')
    ax.set_title(f'{horizon} Returns', color='white')
    
    pct_up = (data > 0).mean()
    ax.text(0.02, 0.95, f'Up: {pct_up:.1%}  Down: {1-pct_up:.1%}',
            transform=ax.transAxes, color='white', fontsize=9, va='top')
    
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2332')

plt.suptitle('Label Distributions by Horizon (Sub-hourly)', color='white', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print('=' * 55)
print('FEATURE ENGINEERING COMPLETE (Sub-hourly)')
print('=' * 55)
print(f'Total rows:     {len(df_all):,}')
print(f'Total features: {len(feat_cols)}')
print(f'Pairs:          {len(PAIRS)}')
print(f'Date range:     {df_all.index[0].date()} -> {df_all.index[-1].date()}')
print(f'\nLabel columns:  {label_cols}')
print(f'\nOutput file:    all_pairs_features_labels_subhourly.parquet')
print(f'File size:      {out_path.stat().st_size / 1024 / 1024:.1f} MB')